# Data-Space Diffusion with AB-UPT on DrivAerML

Single-stage diffusion directly on field values at anchor points.
Same AB-UPT backbone as regression, with three additions:
noisy field projection, timestep conditioning, noise prediction heads.

```
geometry mesh -> SupernodePooling -> geometry blocks -> geometry_encoding
                                                            | (cross-attn)
surface anchors -> pos_embed + bias + field_proj(noisy_fields) + t_emb -> physics blocks -> surface decoder -> noise_head
volume anchors  -> pos_embed + bias + field_proj(noisy_fields) + t_emb -> physics blocks -> volume decoder  -> noise_head
```

see `notebooks/dpf_abupt.md` and `notebooks/dataspace.md` for architecture details.

## pipeline overview

| Stage | What | Output |
|-------|------|--------|
| 1. Train | DiffusionABUPT with flow matching | checkpoint |
| 2. Sample | Iterative denoising from noise to fields | predicted surface + volume fields |
| 3. Evaluate | Per-field MSE vs ground truth | metrics dict |
| 4. Compare | Regression AB-UPT baseline vs diffusion | bar chart + ratio table |

In [ ]:
## Setup

# interactive slurm
# salloc --cpus-per-task=28 --mem=250GB --reservation=dev --gpus-per-node=1 --time 1-0 srun --pty zsh

# start notebook (token "ggall", no password — same URL every time)
# cd ~/exp/noether && source .venv/bin/activate
# jupyter notebook --no-browser --port=8888 --ip=0.0.0.0 --ServerApp.token=ggall --ServerApp.password=''

In [ ]:
import sys

sys.path.insert(0, "/home/ggalletti/exp/noether")
sys.path.insert(0, "/home/ggalletti/exp/noether/diffusion_uq")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from steady_diffusion.experiments import (
    build_abupt_regression_config,
    build_diffusion_ab_upt_config,
)
from steady_diffusion.viz import compute_field_metrics, plot_field_uq_stl

from noether.training.runners import HydraRunner

print(f"pytorch: {torch.__version__}")
print(f"cuda: {torch.cuda.is_available()} ({torch.cuda.device_count()} device(s))")
if torch.cuda.is_available():
    print(f"gpu: {torch.cuda.get_device_name(0)}")
    print(f"memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Config

Both the diffusion model and the regression baseline are trained via SLURM
(`run_abupt_diffusion.sbatch`, `run_abupt_regression.sbatch`). Point the two
`*_CHECKPOINT` paths at the `.th` files — architecture params are parsed from
the `hp_resolved.yaml` sitting next to each run directory so we rebuild the
exact config used at training time.


In [ ]:
DEVICE = "cuda"
DATASET_ROOT = "/nfs-gpu/research/datasets/drivaerml/preprocessed/subsampled_10x"
DATA_SPLIT = "val"

# pretrained checkpoints (.th)
DIFFUSION_CHECKPOINT = "/home/ggalletti/exp/noether/outputs/abupt_diffusion/2026-04-13_a10o8/checkpoints/diffusion_ab_upt_cp=best_model.loss.test.total_model.th"
REGRESSION_CHECKPOINT = "/home/ggalletti/exp/noether/outputs/abupt_regression/18224_2026-04-13_kwx2l/checkpoints/ab_upt_cp=best_model.loss.test.total_model.th"

## Load pretrained diffusion model

Parse `hp_resolved.yaml` next to the checkpoint and rebuild the config via
`build_diffusion_ab_upt_config` so architecture and pipeline match training exactly.


In [ ]:
def _load_hp(ckpt_path: str):
    ckpt = Path(ckpt_path)
    run_dir = ckpt.parent.parent
    hp_path = run_dir / "hp_resolved.yaml"
    assert hp_path.exists(), f"hp_resolved.yaml not found at {hp_path}"
    with open(hp_path) as f:
        hp = yaml.full_load(f)
    return hp, run_dir, ckpt


hp_d, diff_run_dir, diff_ckpt = _load_hp(DIFFUSION_CHECKPOINT)
md_ = hp_d["model"]
pld = hp_d["datasets"]["train"]["pipeline"]
paradigm = hp_d["trainer"]["diffusion"]["paradigm"]

# also parse the regression ckpt so the eval pipeline + model build downstream
# match whatever the reg run was trained at (old 1024/16384 vs new 16384/65536).
hp_r, reg_run_dir, reg_ckpt = _load_hp(REGRESSION_CHECKPOINT)
mr_ = hp_r["model"]
plr = hp_r["datasets"]["train"]["pipeline"]

trainer, model, _, _ = HydraRunner.setup_experiment(
    device=DEVICE,
    config=build_diffusion_ab_upt_config(
        dataset_root=DATASET_ROOT,
        output_path=str(diff_run_dir.parent),
        paradigm=paradigm,
        hidden_dim=md_["hidden_dim"],
        num_heads=md_["transformer_block_config"]["num_heads"],
        mlp_expansion_factor=md_["transformer_block_config"].get("mlp_expansion_factor", 4),
        geometry_depth=md_["geometry_depth"],
        physics_blocks=md_.get("physics_blocks"),
        num_surface_blocks=md_["num_surface_blocks"],
        num_volume_blocks=md_["num_volume_blocks"],
        surface_field_dim=md_["surface_field_dim"],
        volume_field_dim=md_["volume_field_dim"],
        num_geometry_supernodes=pld["num_geometry_supernodes"],
        num_geometry_points=pld["num_geometry_points"],
        num_surface_anchor_points=pld["num_surface_anchor_points"],
        num_volume_anchor_points=pld["num_volume_anchor_points"],
        supernode_radius=md_["supernode_pooling_config"].get("radius", 0.25),
        max_epochs=1,
        batch_size=1,
    ),
)
_state = torch.load(diff_ckpt, map_location=DEVICE, weights_only=False)
model.load_state_dict(_state.get("state_dict", _state))
del _state
model.eval().to(DEVICE)

from steady_diffusion.diffusion import FlowMatchingConfig, FlowMatchingSchedule

if paradigm == "flow_matching":
    schedule = FlowMatchingSchedule(FlowMatchingConfig(minibatch_ot=False)).to(DEVICE)
else:
    raise ValueError(f"add schedule setup for: {paradigm}")

print(f"paradigm: {paradigm}")
print(f"diffusion model loaded — {sum(p.numel() for p in model.parameters()):,} params")

## sampling

Generate fields by iterative denoising. The model takes noisy fields at anchor positions
and predicts the noise (or velocity for flow matching). We denoise surface and volume
fields jointly — the model sees both at each step via the physics blocks.

In [ ]:
# Helpers used by the sampling + eval cells below.


def _get_normalizers(ds):
    """Walk through dataset wrappers to find the field normalizers dict."""
    base = ds
    while hasattr(base, "dataset") and not hasattr(base, "normalizers"):
        base = base.dataset
    return base.normalizers


def denormalize_pred_and_target(pred, batch, normalizers):
    """Denormalize pred dict and matching batch targets to physical space.

    Returns (pred_phys, batch_phys) with denormalized field values.
    """
    pred_phys = {}
    batch_phys = dict(batch)
    for f, v in pred.items():
        if f in normalizers:
            pred_phys[f] = normalizers[f].inverse(v.cpu())
            tgt_key = f"{f}_target"
            if tgt_key in batch:
                batch_phys[tgt_key] = normalizers[f].inverse(batch[tgt_key].cpu())
        else:
            pred_phys[f] = v
    return pred_phys, batch_phys


@torch.no_grad()
def sample_fields_batched(model, schedule, batch, n_samples: int = 1, steps: int = 10):
    """Joint flow-matching sampler.

    Two supported modes:
      - n_samples=1, bs>=1: one draw per geometry in a batched pass. Used to
        batch over geometries with independent UQ draws looped externally.
      - n_samples>1, bs=1:  n_samples parallel draws for a single geometry
        via batch-dim replication (kept for compatibility).

    Caller is responsible for wrapping with enable_geometry_cache /
    disable_geometry_cache — cache then persists across repeated calls on
    the same geometry batch (e.g. the N_UQ loop in the sampling cell).

    Returns surface_stack (B*n_samples, n_surf, 4) + volume_stack (B*n_samples, n_vol, 7).
    """
    B = batch["surface_anchor_position"].shape[0]
    assert n_samples == 1 or B == 1, f"n_samples>1 requires bs=1, got bs={B}, n_samples={n_samples}"

    if n_samples == 1:
        geom_kwargs = {
            "surface_anchor_position": batch["surface_anchor_position"],
            "volume_anchor_position": batch["volume_anchor_position"],
            "geometry_position": batch["geometry_position"],
            "geometry_supernode_idx": batch["geometry_supernode_idx"],
            "geometry_batch_idx": batch["geometry_batch_idx"],
        }
        eff_b = B
    else:

        def rep(x):
            return x.repeat(n_samples, *([1] * (x.dim() - 1))) if torch.is_tensor(x) else x

        sn_idx = batch["geometry_supernode_idx"]
        bi = batch["geometry_batch_idx"]
        n_geom = batch["geometry_position"].shape[0]
        offsets = torch.arange(n_samples, device=sn_idx.device) * n_geom
        geom_kwargs = {
            "surface_anchor_position": rep(batch["surface_anchor_position"]),
            "volume_anchor_position": rep(batch["volume_anchor_position"]),
            "geometry_position": batch["geometry_position"].repeat(n_samples, 1),
            "geometry_supernode_idx": torch.cat([sn_idx + off for off in offsets], dim=0),
            "geometry_batch_idx": torch.cat([torch.full_like(bi, k) for k in range(n_samples)], dim=0),
        }
        eff_b = n_samples

    n_surf = batch["surface_anchor_position"].shape[1]
    n_vol = batch["volume_anchor_position"].shape[1]

    def joint_model_fn(xt_list, t, _cond):
        noisy_surface, noisy_volume = xt_list
        out = model(
            noisy_surface_fields=noisy_surface,
            noisy_volume_fields=noisy_volume,
            timestep=t,
            **geom_kwargs,
        )
        return [out["surface_anchor_noise"], out["volume_anchor_noise"]]

    shapes = [(eff_b, n_surf, 4), (eff_b, n_vol, 7)]
    x_surf, x_vol = schedule.sample_joint(shapes, joint_model_fn, steps=steps)

    return {"surface_stack": x_surf, "volume_stack": x_vol}


def split_fields(stack, field_dims):
    out = {}
    off = 0
    for name, d in field_dims.items():
        out[name] = stack[..., off : off + d]
        off += d
    return out


SURFACE_FIELDS = {"surface_pressure": 1, "surface_friction": 3}
VOLUME_FIELDS = {"volume_pressure": 1, "volume_velocity": 3, "volume_vorticity": 3}

## Full-resolution evaluation

Training used 16K anchors per branch. To check whether the model generalizes
beyond memorized spatial positions, we evaluate on **50K surface + 50K volume**
points — a 3× denser sampling of the same normalized mesh (`positions ∈ [0, 1000]`,
fields z-scored with dataset-wide stats). Pipeline normalization is identical
to training (same `@with_normalizers` on the same `DrivAerMLDataset`).

Both diffusion and regression are evaluated on the **same resolution** so the
comparison is fair.


In [ ]:
# ── Full-resolution eval pipeline ──────────────────────────────────
# Builds the high-res test_dataset used everywhere below. Also a separate
# regression-eval dataset (same eval points, but the regression model has
# its own preset for geometry encoding).
# Pipeline applies: PositionNormalizer (→ [0, 1000]) + MeanStd on fields.
N_EVAL_SURFACE = 50_000
N_EVAL_VOLUME = 50_000

# canonical test dataset for diffusion sampling (high-res, full-mesh eval points)
_eval_diff_tr, _, _, _ = HydraRunner.setup_experiment(
    device="cpu",
    config=build_diffusion_ab_upt_config(
        dataset_root=DATASET_ROOT,
        output_path="./outputs/_eval_tmp",
        paradigm=paradigm,
        hidden_dim=md_["hidden_dim"],
        num_heads=md_["transformer_block_config"]["num_heads"],
        geometry_depth=md_["geometry_depth"],
        physics_blocks=md_.get("physics_blocks"),
        num_surface_blocks=md_["num_surface_blocks"],
        num_volume_blocks=md_["num_volume_blocks"],
        surface_field_dim=md_["surface_field_dim"],
        volume_field_dim=md_["volume_field_dim"],
        num_geometry_supernodes=pld["num_geometry_supernodes"],
        num_geometry_points=pld["num_geometry_points"],
        supernode_radius=md_["supernode_pooling_config"].get("radius", 0.25),
        num_surface_anchor_points=N_EVAL_SURFACE,
        num_volume_anchor_points=N_EVAL_VOLUME,
        max_epochs=1,
        batch_size=1,
    ),
)
test_dataset = _eval_diff_tr.data_container.get_dataset(DATA_SPLIT)
field_normalizers = _get_normalizers(test_dataset)
print(f"normalizers: {list(field_normalizers.keys())}")

# regression eval dataset — same helper the training sbatch uses, so pipeline matches exactly
_eval_reg_tr, _, _, _ = HydraRunner.setup_experiment(
    device="cpu",
    config=build_abupt_regression_config(
        dataset_root=DATASET_ROOT,
        output_path="./outputs/_eval_reg_tmp",
        hidden_dim=mr_["hidden_dim"],
        geometry_depth=mr_["geometry_depth"],
        physics_blocks=mr_.get("physics_blocks"),
        num_geometry_supernodes=plr["num_geometry_supernodes"],
        num_geometry_points=plr["num_geometry_points"],
        num_surface_anchor_points=N_EVAL_SURFACE,
        num_volume_anchor_points=N_EVAL_VOLUME,
        max_epochs=1,
        batch_size=1,
    ),
)
eval_reg_ds = _eval_reg_tr.data_container.get_dataset(DATA_SPLIT)

print(f"eval pipelines ready: {N_EVAL_SURFACE} surface + {N_EVAL_VOLUME} volume points")
print(f"  diffusion test set: {len(test_dataset)} samples")
print(f"  regression test set: {len(eval_reg_ds)} samples")

In [ ]:
def zero_fields(batch):
    """Zero all field tensors so the sampler starts from pure noise — no GT leakage."""
    out = dict(batch)
    for k in out:
        if "geometry" not in k and "position" not in k and torch.is_tensor(out[k]):
            out[k] = torch.zeros_like(out[k])
    return out


def _design_id(ds, idx):
    base = ds
    while hasattr(base, "indices") and hasattr(base, "dataset"):
        idx = base.indices[idx]
        base = base.dataset
    while hasattr(base, "dataset") and not hasattr(base, "design_ids"):
        base = base.dataset
    return int(base.design_ids[idx])


STL_ROOT = "/nfs-gpu/research/datasets/drivaerml/raw_surface_data"
BATCH_GEOM = 16
N_UQ = 5
SAMPLING_STEPS = 10

all_diff_metrics: dict[str, list[float]] = {}
all_samples: list[dict] = []

model.eval()
n_test = len(test_dataset)
for batch_start in range(0, n_test, BATCH_GEOM):
    idxs = list(range(batch_start, min(batch_start + BATCH_GEOM, n_test)))
    B = len(idxs)

    batch = test_dataset.pipeline([test_dataset[i] for i in idxs])
    batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
    sample_batch = zero_fields(batch)
    # print({k: v.shape if isinstance(v, torch.Tensor) else v for k, v in batch.items()})

    import time

    done = min(batch_start + BATCH_GEOM, n_test)
    print(
        f"[diff] batch {batch_start // BATCH_GEOM + 1}/{(n_test + BATCH_GEOM - 1) // BATCH_GEOM} "
        f"(geoms {idxs[0]}..{idxs[-1]}, B={B}) — sampling {N_UQ} draws...",
        flush=True,
    )
    t0 = time.time()

    # Build the geometry cache ONCE for this geometry batch, reuse across N_UQ draws.
    model.enable_geometry_cache()
    per_uq_surf: list[torch.Tensor] = []
    per_uq_vol: list[torch.Tensor] = []
    for u in range(N_UQ):
        tu = time.time()
        r = sample_fields_batched(model, schedule, sample_batch, n_samples=1, steps=SAMPLING_STEPS)
        per_uq_surf.append(r["surface_stack"])  # (B, n_surf, 4)
        per_uq_vol.append(r["volume_stack"])  # (B, n_vol, 7)
    model.disable_geometry_cache()

    surf_all = torch.stack(per_uq_surf, dim=0)  # (N_UQ, B, n_surf, 4)
    vol_all = torch.stack(per_uq_vol, dim=0)  # (N_UQ, B, n_vol, 7)

    # per-geometry metrics + storage
    for j, i in enumerate(idxs):
        surf_stack = surf_all[:, j].cpu()
        vol_stack = vol_all[:, j].cpu()

        design_id = _design_id(test_dataset, i)
        stl_path = f"{STL_ROOT}/run_{design_id}/drivaer_{design_id}.stl"

        # per-geom batch slice for downstream (geometry tensors are flat-concat
        # and not needed post-sampling — keep only anchor-space keys).
        geom_batch = {
            k: v[j : j + 1].cpu()
            for k, v in batch.items()
            if torch.is_tensor(v) and v.dim() >= 1 and v.shape[0] == B and "geometry" not in k
        }

        all_samples.append(
            {
                "surface_stack": surf_stack,
                "volume_stack": vol_stack,
                "batch": geom_batch,
                "design_id": design_id,
                "stl_path": stl_path,
            }
        )

        surf_mean = surf_stack.mean(0, keepdim=True)
        vol_mean = vol_stack.mean(0, keepdim=True)
        pred = {**split_fields(surf_mean, SURFACE_FIELDS), **split_fields(vol_mean, VOLUME_FIELDS)}
        pred_phys, batch_phys = denormalize_pred_and_target(pred, geom_batch, field_normalizers)
        sample_metrics = compute_field_metrics(pred_phys, batch_phys)
        for k, v in sample_metrics.items():
            all_diff_metrics.setdefault(k, []).append(v)

    last_metrics = {k: all_diff_metrics[k][-1] for k in all_diff_metrics}
    print(
        f"[diff] {done}/{n_test} (B={B}, {time.time() - t0:.1f}s total): "
        f"{', '.join(f'{k}={v * 100:.2f}%' for k, v in last_metrics.items())}",
        flush=True,
    )

diff_metrics = {k: float(np.mean(v)) for k, v in all_diff_metrics.items()}
diff_metrics["total_relL2"] = float(np.mean([np.mean(v) for v in all_diff_metrics.values()]))

print(f"\ndiffusion avg per-field rel L2 ({n_test} test geoms, {N_UQ} samples):")
for k, v in sorted(diff_metrics.items()):
    print(f"  {k}: {v * 100:.2f}%")

## Per-point UQ on test samples

For each of `N_SHOW` random test geometries, draw `N_UQ` joint field samples from the
diffusion prior (in a single batched run), then plot GT / mean / std / |error| for
surface pressure and |WSS|.

Pearson correlation between std and |error| (shown in std panel) indicates whether
diffusion variance tracks predictive error.


In [ ]:
N_SHOW = 5

rng = np.random.default_rng(0)
show_idxs = rng.choice(len(all_samples), size=min(N_SHOW, len(all_samples)), replace=False).tolist()
print(f"UQ on test indices: {show_idxs}")

for idx in show_idxs:
    s = all_samples[idx]
    batch = s["batch"]
    print(f"\n=== test[{idx}] run_{s['design_id']} ===")

    surface_samples = split_fields(s["surface_stack"].numpy(), SURFACE_FIELDS)
    positions = batch["surface_anchor_position"][0].numpy()

    sp = surface_samples["surface_pressure"].squeeze(-1)
    sp_target = batch["surface_pressure_target"][0].numpy().squeeze(-1)
    plot_field_uq_stl(
        positions, sp_target, sp.mean(0), sp.std(0), stl_path=s["stl_path"], field_name="Cp", index=idx, view="front"
    )

    sf_mag = np.linalg.norm(surface_samples["surface_friction"], axis=-1)
    sf_target = np.linalg.norm(batch["surface_friction_target"][0].numpy(), axis=-1)
    plot_field_uq_stl(
        positions,
        sf_target,
        sf_mag.mean(0),
        sf_mag.std(0),
        stl_path=s["stl_path"],
        field_name="|WSS|",
        index=idx,
        view="front",
    )

## UQ on integrated quantities

For each of `N_CAL` test geometries, draw `N_UQ_INT` diffusion samples (batched),
compute scalar summaries, and plot **target vs. mean ± std** with calibration metrics:

- **R²** of predicted mean vs target
- **coverage @ 1σ** (frac within mean ± 1·std) — ideal ≈ 0.68
- **mean |z-score|** — ideal ≈ 0.8 (half-normal)


In [ ]:
SCALARS = {
    "mean Cp": lambda cp, wss: cp.mean(),
    "max Cp": lambda cp, wss: cp.max(),
    "min Cp": lambda cp, wss: cp.min(),
    "mean |WSS|": lambda cp, wss: wss.mean(),
    "max |WSS|": lambda cp, wss: wss.max(),
}

N_CAL = min(20, len(all_samples))
rng = np.random.default_rng(1)
cal_idxs = rng.choice(N_CAL, size=N_CAL, replace=False).tolist()
print(f"integrated-quantity UQ on {N_CAL} test geometries (from stored samples)")

targets = {k: [] for k in SCALARS}
pred_means = {k: [] for k in SCALARS}
pred_stds = {k: [] for k in SCALARS}

for j, idx in enumerate(cal_idxs):
    s = all_samples[idx]
    batch = s["batch"]

    # GT from stored batch
    cp_t = batch["surface_pressure_target"][0].numpy().squeeze(-1)
    wss_t = np.linalg.norm(batch["surface_friction_target"][0].numpy(), axis=-1)
    for name, fn in SCALARS.items():
        targets[name].append(float(fn(cp_t, wss_t)))

    # scalars from stored diffusion samples
    surf = split_fields(s["surface_stack"].numpy(), SURFACE_FIELDS)
    cp_s = surf["surface_pressure"].squeeze(-1)
    wss_s = np.linalg.norm(surf["surface_friction"], axis=-1)

    for name, fn in SCALARS.items():
        per = np.array([float(fn(cp_s[k], wss_s[k])) for k in range(cp_s.shape[0])])
        pred_means[name].append(float(per.mean()))
        pred_stds[name].append(float(per.std()))

    if (j + 1) % 5 == 0 or j == N_CAL - 1:
        print(f"  [{j + 1}/{N_CAL}]")

In [ ]:
fig, axes = plt.subplots(1, len(SCALARS), figsize=(5 * len(SCALARS), 5))

print(f"\n{'quantity':<14} {'R^2':>8} {'cov@1sig':>10} {'|z|':>8}")
print("-" * 42)

for ax, name in zip(axes, SCALARS, strict=True):
    t = np.array(targets[name])
    mu = np.array(pred_means[name])
    sd = np.array(pred_stds[name])
    ss_res = float(((t - mu) ** 2).sum())
    ss_tot = float(((t - t.mean()) ** 2).sum())
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    eps = 1e-12
    within = float(np.mean(np.abs(t - mu) <= sd))
    zabs = float(np.mean(np.abs(t - mu) / (sd + eps)))

    ax.errorbar(t, mu, yerr=sd, fmt="o", ms=5, alpha=0.7, capsize=3, color="tab:blue")
    lo = float(min(t.min(), (mu - sd).min()))
    hi = float(max(t.max(), (mu + sd).max()))
    ax.plot([lo, hi], [lo, hi], "k--", alpha=0.5, label="y=x")
    ax.set_xlabel(f"target {name}")
    ax.set_ylabel(f"pred {name} (mean ± std)")
    ax.set_title(f"{name}\nR²={r2:.3f}, cov@1σ={within:.2f}")
    ax.legend(loc="best", fontsize=8)
    ax.grid(alpha=0.3)
    print(f"{name:<14} {r2:>8.3f} {within:>10.2f} {zabs:>8.2f}")

plt.tight_layout()
plt.show()
print("\nideal: cov@1 ≈ 0.68, |z| ≈ 0.80 (half-normal)")

## regression baseline: plain AB-UPT

A standard supervised AB-UPT that predicts surface + volume fields directly from geometry —
no latent bottleneck, no noisy field input. Strict baseline for the diffusion model.

Uses noether's built-in `AeroABUPT` (via `build_abupt_regression_config`).


In [ ]:
# regression baseline: load model. Eval pipeline (eval_reg_ds) was built in cell 10.
# hp_r / reg_run_dir / reg_ckpt / mr_ / plr were loaded in cell 6 alongside hp_d.

# patch AeroABUPT.forward: backbone returns (preds, kv_cache) post-PR#131
from noether.modeling.models.aerodynamics import AeroABUPT

_orig_abupt_fwd = AeroABUPT.forward


def _abupt_forward_unwrap(self, **kwargs):
    out = _orig_abupt_fwd(self, **kwargs)
    return out[0] if isinstance(out, tuple) else out


AeroABUPT.forward = _abupt_forward_unwrap

reg_trainer, reg_model, _, _ = HydraRunner.setup_experiment(
    device=DEVICE,
    config=build_abupt_regression_config(
        dataset_root=DATASET_ROOT,
        output_path=str(reg_run_dir.parent),
        hidden_dim=mr_["hidden_dim"],
        geometry_depth=mr_["geometry_depth"],
        physics_blocks=mr_.get("physics_blocks"),
        num_geometry_supernodes=plr["num_geometry_supernodes"],
        num_geometry_points=plr["num_geometry_points"],
        num_surface_anchor_points=N_EVAL_SURFACE,
        num_volume_anchor_points=N_EVAL_VOLUME,
        max_epochs=1,
        batch_size=1,
    ),
)
_reg_state = torch.load(reg_ckpt, map_location=DEVICE, weights_only=False)
reg_model.load_state_dict(_reg_state.get("state_dict", _reg_state))
del _reg_state
reg_model.eval().to(DEVICE)

print(f"regression model loaded — {sum(p.numel() for p in reg_model.parameters()):,} params")
print(f"eval pipeline: {N_EVAL_SURFACE} surface + {N_EVAL_VOLUME} volume anchors")

In [ ]:
# evaluate regression AB-UPT on full-res eval points (deterministic, no sampling)
reg_all: dict[str, list[float]] = {}
reg_model.eval()
with torch.no_grad():
    for i in range(len(eval_reg_ds)):
        batch = eval_reg_ds.pipeline([eval_reg_ds[i]])
        batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        n_surf = batch["surface_anchor_position"].shape[1]
        n_vol = batch["volume_anchor_position"].shape[1]

        pred = reg_model(
            geometry_position=batch["geometry_position"],
            geometry_supernode_idx=batch["geometry_supernode_idx"],
            geometry_batch_idx=batch["geometry_batch_idx"],
            surface_anchor_position=batch["surface_anchor_position"],
            volume_anchor_position=batch["volume_anchor_position"],
        )
        pred_phys, batch_phys = denormalize_pred_and_target(pred, batch, field_normalizers)
        m = compute_field_metrics(pred_phys, batch_phys)
        for k, v in m.items():
            reg_all.setdefault(k, []).append(v)
        if (i + 1) % 5 == 0 or i == 0:
            print(
                f"[reg] geom {i + 1}/{len(eval_reg_ds)} ({n_surf}s+{n_vol}v pts): "
                f"{', '.join(f'{k}={v:.6f}' for k, v in m.items())}"
            )

reg_metrics = {k: float(np.mean(v)) for k, v in reg_all.items()}
reg_metrics["total_relL2"] = float(np.mean([np.mean(v) for v in reg_all.values()]))

print(f"\nregression AB-UPT avg per-field rel L2 ({N_EVAL_SURFACE}s+{N_EVAL_VOLUME}v eval pts):")
for k, v in sorted(reg_metrics.items()):
    print(f"  {k}: {v * 100:.2f}%")

### comparison: diffusion vs regression

In [ ]:
import matplotlib.pyplot as plt

fields = [k for k in sorted(diff_metrics) if k != "total_relL2" and k in reg_metrics]

fig, ax = plt.subplots(figsize=(max(6, len(fields) * 1.3), 4))
x = np.arange(len(fields))
w = 0.38
ax.bar(x - w / 2, [reg_metrics[f] * 100 for f in fields], w, label="regression AB-UPT", color="tab:blue")
ax.bar(x + w / 2, [diff_metrics[f] * 100 for f in fields], w, label="diffusion", color="tab:orange")
ax.set_xticks(x)
ax.set_xticklabels(fields, rotation=25, ha="right")
ax.set_ylabel("rel L2 (%)")
ax.set_title(f"per-field rel L2 on {len(eval_reg_ds)} test geoms ({N_EVAL_SURFACE}s+{N_EVAL_VOLUME}v pts)")
ax.grid(alpha=0.3, axis="y")
ax.legend()

print(f"\n{'field':<22} {'regression':>12} {'diffusion':>12} {'ratio (diff/reg)':>18}")
print("-" * 68)
for f in fields + (["total_relL2"] if "total_relL2" in reg_metrics else []):
    a, d = reg_metrics[f], diff_metrics[f]
    print(f"{f:<22} {a:>12.6f} {d:>12.6f} {d / a:>18.2f}")